# Dataset AniList : features des animes (v1)

Une ligne par média anime (une saison = une ligne, comme sur AniList), contenu adulte exclu,
uniquement les médias **terminés avant novembre 2023** (fin en année + mois).

Étapes :
1. Configuration
2. Client GraphQL (débit limité, gestion des 429, cache JSON des pages brutes)
3. Collecte, découpée par année de fin
4. Aplatissement et ingénierie des features (dates, titre, description, genres, tags)
5. Filtre temporel et export CSV
6. Aperçu du dataset

In [1]:
import html
import json
import re
import time
from collections import Counter
from pathlib import Path

import pandas as pd
import requests

## 1. Configuration

In [2]:
URL = "https://graphql.anilist.co"

# Coupure temporelle : on garde les médias dont (année, mois) de fin < (2023, 11)
CUTOFF_YEAR, CUTOFF_MONTH = 2023, 11

# Débit : l'API est actuellement limitée à 30 requêtes/min -> une requête toutes les 2,5 s
MIN_INTERVAL_S = 2.5
PER_PAGE = 50  # maximum autorisé par AniList

# Découpage de la collecte : un premier lot "tout ce qui finit avant FIRST_YEAR", puis un lot par année
FIRST_YEAR = 1970

# Tags : on garde ceux dont le rang (pertinence 0-100 votée par la communauté) >= TAG_MIN_RANK,
# puis seulement les TOP_N_TAGS plus fréquents du dataset
TAG_MIN_RANK = 50
TOP_N_TAGS = 50

CACHE_DIR = Path("cache_anilist")
CACHE_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = Path("anilist_anime_features.csv")

# Motifs cherchés dans les titres (anglais + romaji), insensibles à la casse, avec limites de mot
FINAL_PATTERN = re.compile(r"\b(final|end|ending|kanketsu(?:-?hen)?)\b", re.IGNORECASE)
SEASON_PATTERN = re.compile(r"\b(season|saison|part|cour|\d+(?:st|nd|rd|th))\b", re.IGNORECASE)
# Chiffres romains en fin de titre ("Overlord II") : sensibles à la casse pour éviter les faux positifs
ROMAN_SUFFIX_PATTERN = re.compile(r"\s(?:II|III|IV|V|VI|VII|VIII|IX|X)$")

## 2. Client GraphQL

In [3]:
# Réutilise la même session HTTP pour toutes les requêtes vers AniList.
session = requests.Session()
session.headers.update({"Content-Type": "application/json", "Accept": "application/json"})
_last_request_t = 0.0


def post_graphql(query, variables, max_retries=6):
    """POST une requête GraphQL en respectant MIN_INTERVAL_S, avec gestion des 429 et des erreurs serveur."""
    # Cette variable globale permet de mesurer le délai depuis la dernière requête.
    global _last_request_t
    for attempt in range(max_retries):
        # Respecte l'intervalle minimal entre deux requêtes successives.
        wait = MIN_INTERVAL_S - (time.monotonic() - _last_request_t)
        if wait > 0:
            time.sleep(wait)
        _last_request_t = time.monotonic()

        try:
            # Le timeout évite de rester bloqué indéfiniment sur une requête.
            r = session.post(URL, json={"query": query, "variables": variables}, timeout=30)
        except requests.RequestException as e:
            # En cas d'erreur réseau, applique un délai exponentiel avant de réessayer.
            backoff = 5 * 2 ** attempt
            print(f"  erreur réseau ({e}), nouvel essai dans {backoff} s")
            time.sleep(backoff)
            continue

        if r.status_code == 429:
            # AniList indique éventuellement la durée à attendre dans Retry-After.
            retry_after = int(r.headers.get("Retry-After", 60)) + 1
            print(f"  429 Too Many Requests, attente {retry_after} s")
            time.sleep(retry_after)
            continue
        if r.status_code >= 500:
            # Les erreurs serveur peuvent être temporaires : on réessaie avec backoff.
            backoff = 5 * 2 ** attempt
            print(f"  erreur serveur {r.status_code}, nouvel essai dans {backoff} s")
            time.sleep(backoff)
            continue

        # Les erreurs HTTP non récupérables et les erreurs GraphQL sont remontées.
        data = r.json()
        if r.status_code != 200 or data.get("errors"):
            raise RuntimeError(f"Erreur GraphQL (HTTP {r.status_code}) : {data.get('errors')}")
        return data

    # Signale l'échec quand toutes les tentatives ont été consommées.
    raise RuntimeError(f"Échec après {max_retries} tentatives")

## 3. Collecte

On découpe par **année de fin** (`endDate_greater` / `endDate_lesser`, au format FuzzyDateInt `AAAAMMJJ`) :
les pages restent courtes, et une interruption ne coûte que le lot en cours.
Chaque page brute est sauvegardée dans `cache_anilist/` : relancer la cellule reprend là où on s'est arrêté
sans refaire les requêtes déjà faites. Supprimer le dossier pour tout recollecter.

Les médias sans date de fin ne sont pas renvoyés par le filtre `endDate_lesser`, ce qui nous convient.

In [6]:
QUERY = """
query ($page: Int, $perPage: Int, $endGreater: FuzzyDateInt, $endLesser: FuzzyDateInt) {
  Page(page: $page, perPage: $perPage) {
    pageInfo { currentPage hasNextPage }
    media(type: ANIME, isAdult: false, endDate_greater: $endGreater, endDate_lesser: $endLesser, sort: ID) {
      id
      title { romaji english }
      format
      description(asHtml: false)
      startDate { year month }
      endDate { year month }
      season
      seasonYear
      episodes
      duration
      countryOfOrigin
      source(version: 3)
      hashtag
      genres
      tags { name rank isAdult }
    }
  }
}
"""


def make_buckets():
    """Lots (nom, endDate_greater, endDate_lesser). Les bornes sont strictes."""
    buckets = [(f"avant_{FIRST_YEAR}", 10000000, FIRST_YEAR * 10000)]
    for y in range(FIRST_YEAR, CUTOFF_YEAR + 1):
        buckets.append((str(y), y * 10000 - 1, (y + 1) * 10000))
    return buckets


def fetch_bucket(name, end_greater, end_lesser):
    media, page = [], 1
    while True:
        cache_file = CACHE_DIR / f"{name}_p{page:03d}.json"
        if cache_file.exists():
            data = json.loads(cache_file.read_text(encoding="utf-8"))
        else:
            variables = {"page": page, "perPage": PER_PAGE,
                         "endGreater": end_greater, "endLesser": end_lesser}
            data = post_graphql(QUERY, variables)
            cache_file.write_text(json.dumps(data, ensure_ascii=False), encoding="utf-8")
        page_data = data["data"]["Page"]
        media.extend(page_data["media"])
        if not page_data["pageInfo"]["hasNextPage"]:
            return media
        page += 1


def collect_all():
    all_media = []
    t0 = time.time()
    for name, g, l in make_buckets():
        m = fetch_bucket(name, g, l)
        all_media.extend(m)
        print(f"{name:>12} : {len(m):5d} médias  (total {len(all_media):6d}, {time.time() - t0:6.0f} s)")
    return all_media

In [7]:
raw_media = collect_all()
print(f"\n{len(raw_media)} médias récupérés")

  avant_1970 :   576 médias  (total    576,     28 s)
        1970 :    47 médias  (total    623,     30 s)
        1971 :    48 médias  (total    671,     33 s)
        1972 :    50 médias  (total    721,     38 s)
        1973 :    32 médias  (total    753,     40 s)
        1974 :    47 médias  (total    800,     43 s)
        1975 :    49 médias  (total    849,     45 s)
        1976 :    49 médias  (total    898,     48 s)
        1977 :    49 médias  (total    947,     50 s)
        1978 :    66 médias  (total   1013,     55 s)
        1979 :    91 médias  (total   1104,     60 s)
        1980 :    90 médias  (total   1194,     65 s)
        1981 :    96 médias  (total   1290,     70 s)
        1982 :   106 médias  (total   1396,     78 s)
        1983 :   105 médias  (total   1501,     85 s)
        1984 :   121 médias  (total   1622,     93 s)
  429 Too Many Requests, attente 59 s
        1985 :   129 médias  (total   1751,    159 s)
        1986 :   145 médias  (total   1896, 

## 4. Aplatissement et features

In [8]:
def clean_description(text):
    if not text:
        return None
    text = re.sub(r"<[^>]+>", " ", text)       # balises résiduelles (<br>, <i>...)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def title_flags(english, romaji):
    titles = [t for t in (english, romaji) if t]
    has_final = any(FINAL_PATTERN.search(t) for t in titles)
    has_season = any(SEASON_PATTERN.search(t) or ROMAN_SUFFIX_PATTERN.search(t) for t in titles)
    return has_final, has_season


def slug(name):
    return re.sub(r"[^0-9a-z]+", "_", name.lower()).strip("_")


def kept_tags(media):
    return [t["name"] for t in (media.get("tags") or [])
            if not t.get("isAdult") and (t.get("rank") or 0) >= TAG_MIN_RANK]


def flatten(media):
    title = media.get("title") or {}
    start = media.get("startDate") or {}
    end = media.get("endDate") or {}
    has_final, has_season = title_flags(title.get("english"), title.get("romaji"))
    return {
        "id": media["id"],
        "title_romaji": title.get("romaji"),
        "title_english": title.get("english"),
        "title_has_final": has_final,
        "title_has_season": has_season,
        "format": media.get("format"),
        "description": clean_description(media.get("description")),
        "start_year": start.get("year"),
        "start_month": start.get("month"),
        "end_year": end.get("year"),
        "end_month": end.get("month"),
        "season": media.get("season"),
        "seasonYear": media.get("seasonYear"),
        "episodes": media.get("episodes"),
        "duration": media.get("duration"),
        "countryOfOrigin": media.get("countryOfOrigin"),
        "source": media.get("source"),
        "has_hashtag": bool(media.get("hashtag")),
        "_genres": media.get("genres") or [],
        "_tags": kept_tags(media),
    }


df = pd.DataFrame([flatten(m) for m in raw_media]).drop_duplicates("id").set_index("id").sort_index()

int_cols = ["start_year", "start_month", "end_year", "end_month", "seasonYear", "episodes", "duration"]
df[int_cols] = df[int_cols].astype("Int64")   # entiers nullables : une case vide reste vide (pas de float)
for col in ["format", "season", "countryOfOrigin", "source"]:
    df[col] = df[col].astype("category")

### Filtre temporel

On garde un média si sa fin est connue et strictement antérieure à novembre 2023 :
- fin avant 2023 : gardé, même si le mois est inconnu ;
- fin en 2023 : gardé seulement si le mois est connu et < 11 ;
- le reste (fin fin 2023, ou 2023 sans mois) est exclu.

In [9]:
before_cutoff_year = df["end_year"] < CUTOFF_YEAR
in_cutoff_year_ok = (df["end_year"] == CUTOFF_YEAR) & (df["end_month"] < CUTOFF_MONTH)
mask = (before_cutoff_year | in_cutoff_year_ok).fillna(False).astype(bool)

print(f"Avant filtre : {len(df)} | exclus : {(~mask).sum()} | gardés : {mask.sum()}")
df = df[mask].copy()

Avant filtre : 17953 | exclus : 171 | gardés : 17782


### Genres et tags en multi-hot

Une colonne booléenne par genre (`genre_...`), plus `n_genres`.
Pour les tags : seuil de pertinence `TAG_MIN_RANK`, puis les `TOP_N_TAGS` plus fréquents **après** filtre temporel (`tag_...`).

In [10]:
all_genres = sorted({g for gs in df["_genres"] for g in gs})
genre_cols = pd.DataFrame(
    {f"genre_{slug(g)}": df["_genres"].apply(lambda gs, g=g: g in gs) for g in all_genres},
    index=df.index,
)

tag_counts = Counter(t for ts in df["_tags"] for t in ts)
top_tags = [t for t, _ in tag_counts.most_common(TOP_N_TAGS)]
tag_cols = pd.DataFrame(
    {f"tag_{slug(t)}": df["_tags"].apply(lambda ts, t=t: t in ts) for t in top_tags},
    index=df.index,
)

df["n_genres"] = df["_genres"].apply(len)
df = pd.concat([df.drop(columns=["_genres", "_tags"]), genre_cols, tag_cols], axis=1)

print(f"{len(all_genres)} genres, {len(top_tags)} tags gardés (sur {len(tag_counts)} au-dessus du seuil)")
print("Tags gardés :", ", ".join(top_tags))

18 genres, 50 tags gardés (sur 355 au-dessus du seuil)
Tags gardés : Male Protagonist, Female Protagonist, School, Primarily Female Cast, Shounen, Magic, Ensemble Cast, Kids, Heterosexual, Tragedy, Super Power, Primarily Adult Cast, Animals, Primarily Male Cast, Urban Fantasy, Episodic, Urban, Primarily Teen Cast, Historical, Seinen, Parody, Nudity, Military, Coming of Age, Swordplay, Cute Girls Doing Cute Things, CGI, Guns, Demons, Henshin, Foreign, War, Space, Female Harem, Aliens, Full CGI, Kuudere, Gore, Surreal Comedy, Robots, Idol, Isekai, Slapstick, Philosophy, Love Triangle, School Club, Chibi, Work, Tsundere, Shoujo


## 5. Export CSV

In [11]:
df.to_csv(OUTPUT_CSV, encoding="utf-8")
print(f"Écrit : {OUTPUT_CSV.resolve()}  ({df.shape[0]} lignes x {df.shape[1]} colonnes)")

Écrit : /home/bombastus/Documents/INFOIA/PIIA1/projet_PIIA_ecn-/anilist_anime_features.csv  (17782 lignes x 86 colonnes)


## 6. Aperçu du dataset

Remarque : le CSV ne conserve pas les types. Pour relire avec les bons types :
`pd.read_csv(OUTPUT_CSV, index_col="id", dtype={c: "Int64" for c in int_cols})`.

In [12]:
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
df.head()

,title_romaji,title_english,title_has_final,title_has_season,format,description,start_year,start_month,end_year,end_month,season,seasonYear,episodes,duration,countryOfOrigin,...,tag_full_cgi,tag_kuudere,tag_gore,tag_surreal_comedy,tag_robots,tag_idol,tag_isekai,tag_slapstick,tag_philosophy,tag_love_triangle,tag_school_club,tag_chibi,tag_work,tag_tsundere,tag_shoujo
id,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,Cowboy Bebop,Cowboy Bebop,False,False,TV,"Enter a world in the distant future, where Bounty Hunter...",1998,4,1999,4,SPRING,1998,26,24,JP,...,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False
5,Cowboy Bebop: Tengoku no Tobira,Cowboy Bebop: The Movie - Knockin' on Heaven's Door,False,False,MOVIE,"As the Cowboy Bebop crew travels the stars, they learn o...",2001,9,2001,9,SUMMER,2001,1,115,JP,...,False,False,False,False,False,False,False,False,True,False,False,False,True,False,False
6,TRIGUN,Trigun,False,False,TV,Vash the Stampede is a wanted man with a habit of turnin...,1998,4,1998,9,SPRING,1998,26,24,JP,...,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False
7,Witch Hunter ROBIN,Witch Hunter ROBIN,False,False,TV,Robin Sena is a powerful craft user drafted into the STN...,2002,7,2002,12,SUMMER,2002,26,25,JP,...,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
8,Bouken Ou Beet,Beet the Vandel Buster,False,False,TV,It is the dark century and the people are suffering unde...,2004,9,2005,9,FALL,2004,52,23,JP,...,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False


In [13]:
# Colonnes "principales" (hors booléens genre_/tag_, trop nombreux pour être lisibles ici)
main_cols = [c for c in df.columns if not c.startswith(("genre_", "tag_"))]
df[main_cols].info(show_counts=True)
print(f"\n+ {len(genre_cols.columns)} colonnes genre_* et {len(tag_cols.columns)} colonnes tag_*, toutes de type bool sans case vide")

<class 'pandas.DataFrame'>
Index: 17782 entries, 1 to 219626
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   title_romaji      17782 non-null  str     
 1   title_english     8565 non-null   str     
 2   title_has_final   17782 non-null  bool    
 3   title_has_season  17782 non-null  bool    
 4   format            17781 non-null  category
 5   description       16769 non-null  str     
 6   start_year        17782 non-null  Int64   
 7   start_month       16104 non-null  Int64   
 8   end_year          17782 non-null  Int64   
 9   end_month         16023 non-null  Int64   
 10  season            11396 non-null  category
 11  seasonYear        11396 non-null  Int64   
 12  episodes          17780 non-null  Int64   
 13  duration          17656 non-null  Int64   
 14  countryOfOrigin   17782 non-null  category
 15  source            15637 non-null  category
 16  has_hashtag       17782 non-null  boo

In [14]:
# Cases vides par colonne (uniquement celles qui en ont)
missing = pd.DataFrame({
    "n_vides": df.isna().sum(),
    "pct_vides": (df.isna().mean() * 100).round(1),
    "dtype": df.dtypes.astype(str),
})
missing[missing["n_vides"] > 0].sort_values("n_vides", ascending=False)

,n_vides,pct_vides,dtype
title_english,9217,51.8,str
season,6386,35.9,category
seasonYear,6386,35.9,Int64
source,2145,12.1,category
end_month,1759,9.9,Int64
start_month,1678,9.4,Int64
description,1013,5.7,str
duration,126,0.7,Int64
episodes,2,0.0,Int64
format,1,0.0,category


In [15]:
# Statistiques des colonnes numériques
df[int_cols + ["n_genres"]].describe().T

,count,mean,std,min,25%,50%,75%,max
start_year,17782.0,2006.635024,15.809239,1907.0,2000.0,2012.0,2018.0,2023.0
start_month,16104.0,6.370156,3.422662,1.0,4.0,7.0,10.0,12.0
end_year,17782.0,2006.874873,15.756917,1907.0,2001.0,2012.0,2018.0,2023.0
end_month,16023.0,6.692005,3.454404,1.0,3.0,7.0,9.0,12.0
seasonYear,11396.0,2006.829063,12.945462,1940.0,2000.0,2010.0,2017.0,2023.0
episodes,17780.0,11.894938,38.928822,1.0,1.0,1.0,12.0,1818.0
duration,17656.0,23.414137,26.130841,1.0,4.0,20.0,25.0,168.0
n_genres,17782.0,2.026319,1.447149,0.0,1.0,2.0,3.0,9.0


In [16]:
# Répartition par format, source et pays
for col in ["format", "source", "countryOfOrigin"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())


--- format ---
format
TV          4118
MOVIE       3719
MUSIC       2471
ONA         2397
OVA         2260
SPECIAL     1712
TV_SHORT    1104
NaN            1

--- source ---
source
ORIGINAL              7242
MANGA                 4353
NaN                   2145
VIDEO_GAME             947
LIGHT_NOVEL            808
OTHER                  807
NOVEL                  506
VISUAL_NOVEL           326
WEB_NOVEL              255
MULTIMEDIA_PROJECT     202
PICTURE_BOOK            85
ANIME                   42
DOUJINSHI               20
LIVE_ACTION             19
COMIC                   16
GAME                     9

--- countryOfOrigin ---
countryOfOrigin
JP    16127
CN     1286
KR      340
TW       29


In [17]:
# Fréquence des genres et des booléens de titre (en %)
flag_cols = ["title_has_final", "title_has_season", "has_hashtag"]
(df[flag_cols + list(genre_cols.columns)].mean() * 100).round(1).sort_values(ascending=False)

genre_comedy           35.0
genre_action           24.8
genre_fantasy          21.5
has_hashtag            19.0
genre_adventure        18.9
genre_drama            16.4
genre_sci_fi           15.4
genre_slice_of_life    13.9
genre_romance          12.2
genre_supernatural      8.8
genre_mecha             6.2
genre_psychological     5.0
genre_ecchi             5.0
genre_mystery           4.7
genre_music             4.4
genre_sports            4.4
title_has_season        3.8
genre_horror            2.7
genre_mahou_shoujo      2.3
genre_thriller          1.2
title_has_final         0.6
dtype: float64